# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [16]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Inspect the real schema before writing any query against it
con.sql(f"SELECT * FROM {MONTH} LIMIT 3").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


**One row.** In `fact_content_daily_performance`, one row = one content item on one
report date (`client_hash_id` x `content_hash_id` x `report_date`). My lane
aggregates that to one row per content item, so the modelling grain is one page.

**Tables.** fact_content_daily_performance for the grain, the three verification queries, and all five features; dim_clients for per-client history coverage in the limitation.

**Time window.** I develop on `month=2026-03`, a mid-panel month. I avoid the
`_sample` table because it is exactly the final month (June 2026) — the natural
outcome window for any past-to-future label. Developing label logic there would
mean tuning against my own answer key, so the final month stays sealed.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I would predict.** A decline in impressions, defined on a window that comes
strictly after my feature window. This is a proxy for "this page needs a reviewer's
attention", not an observed editorial outcome.

**Field buckets.**
- Label: impressions measured in the outcome window.
- Features: impressions, clicks, average position, days visible, click-through rate —
  all measured before the outcome window.
- Context (splitting and filtering only, never features): `client_hash_id`,
  `content_hash_id`, `report_date`, `ga4_data_available`.
- Excluded: anything measured inside the outcome window.

**What I deliberately exclude, and why.** Any column that only exists after the
outcome has happened. A page's own outcome cannot be an input to predicting that
outcome. Section 3 demonstrates what happens when I break this rule on purpose.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
con.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS distinct_keys
    FROM {MONTH}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,distinct_keys
0,9841378,9841378


This query confirms the primary key grain: the total row count matches the count of distinct (client_hash_id, content_hash_id, report_date) tuples exactly (9,841,378 rows). This proves there are no duplicate records per page per day.

In [18]:
con.sql(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT content_hash_id) AS content_items,
           COUNT(DISTINCT client_hash_id) AS clients,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day
    FROM {MONTH}
""").df()

,rows,content_items,clients,first_day,last_day
0,9841378,331437,55,2026-03-01,2026-03-31


This query verifies the active slice volume for the development month (2026-03). It covers 331,437 unique content items across 55 active clients, spanning exactly 31 days from 2026-03-01 to 2026-03-31 with a total of 9,841,378 daily performance records

In [19]:
con.sql(f"""
    SELECT COUNT(*) AS all_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS ga4_pct
    FROM {MONTH}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,ga4_rows,ga4_pct
0,9841378,413966,4.2


Only 4.2% of March rows (413,966 of 9,841,378) have `ga4_data_available IS TRUE`.
Any GA4-derived feature would therefore be missing for ~96% of my slice — which is
why all five of my features come from GSC columns only.

In [20]:
# --- Section 3: Feature Frame Construction ---

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {MONTH}),
    agg AS (
        SELECT f.client_hash_id,
               f.content_hash_id,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS imp_early,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY
                        THEN f.gsc_clicks ELSE 0 END) AS clicks_early,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY
                        THEN f.gsc_avg_position END) AS pos_early,
               COUNT(DISTINCT CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY
                                    AND f.gsc_impressions > 0
                                   THEN f.report_date END) AS days_visible_early,
               ROUND(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY THEN f.gsc_clicks ELSE 0 END)::FLOAT /
                     NULLIF(SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 10 DAY THEN f.gsc_impressions ELSE 0 END), 0), 4) AS ctr_early,
               SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 10 DAY
                        THEN f.gsc_impressions ELSE 0 END) AS imp_outcome
        FROM {MONTH} f, bounds b
        GROUP BY 1, 2
    )
    SELECT * FROM agg WHERE imp_early >= 50
""").df()

print(f"Total features frame rows: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total features frame rows: 104,039


,client_hash_id,content_hash_id,imp_early,clicks_early,pos_early,days_visible_early,ctr_early,imp_outcome
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,63.0,0.0,3.568254,18,0.0000,14.0
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,309.0,2.0,4.553920,19,0.0065,293.0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,611.0,1.0,4.348539,19,0.0016,199.0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,63.0,0.0,7.828241,18,0.0000,19.0
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1188.0,4.0,1.948582,20,0.0034,670.0


| Feature | Knowable at the decision moment because... |
|---|---|
| `imp_early` | it sums impressions only from days before the outcome window opens |
| `clicks_early` | same window as above; clicks are recorded the day they happen |
| `pos_early` | averages position over the pre-decision days only |
| `days_visible_early` | counts days with impressions inside the pre-decision window |
| `ctr_early` | both inputs come from the pre-decision window, so the ratio is knowable before the outcome period opens |

In [25]:
# --- Section 3: Leakage Experiment ---

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = features.dropna().copy()
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
df["is_declining"] = (df["imp_outcome"] < 0.8 * df["imp_early"]).astype(int)

honest = ["imp_early", "clicks_early", "pos_early", "days_visible_early", "ctr_early"]
leaky = honest + ["imp_outcome"]

def quick_auc(cols):
    X_tr, X_te, y_tr, y_te = train_test_split(
        df[cols], df["is_declining"], test_size=0.3, random_state=42, stratify=df["is_declining"]
    )
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

auc_honest = quick_auc(honest)
auc_leaky = quick_auc(leaky)

print(f"Honest features        : ROC AUC {auc_honest:.3f}")
print(f"With imp_outcome added : ROC AUC {auc_leaky:.3f}")

Honest features        : ROC AUC 0.704
With imp_outcome added : ROC AUC 0.999


**The leak, and its removal.** With the five honest features the model scores
ROC AUC 0.704. Adding `imp_outcome` pushes it to 0.999 — essentially perfect.
That is not skill: `is_declining` is defined as `imp_outcome < 0.8 * imp_early`,
so the model is reading its own answer key. I drop the column and report 0.704
as the only honest number.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [22]:
con.sql(f"""
    SELECT
        MIN(gsc_data_start) AS earliest_client_start,
        MAX(gsc_data_start) AS latest_client_start,
        COUNT(DISTINCT client_hash_id) AS total_clients
    FROM read_parquet('{REL}/dim_clients.parquet')
""").df()

,earliest_client_start,latest_client_start,total_clients
0,2025-01-27,2026-06-02,104




Clients enter the warehouse at different times: the earliest gsc_data_start is 2025-01-27 and the latest is 2026-06-02, a spread of roughly 17 months across 104 clients. A page with few active days in March may therefore be newly registered rather than genuinely low-visibility, and within a single calendar month I cannot tell those two cases apart.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.